# Generative AI Applications - Conditional DCGAN on Fashion-MNIST

## How this notebook is organised

1. Imports.
2. Task definition and model choice.

> TODO: extend this index as the sections land.


## 1. Imports


In [1]:
import hashlib
import math
import random
import time
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch import optim
from torch.utils.data import DataLoader
import torchvision.datasets as datasets
import torchvision.transforms as transforms

import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"torch {torch.__version__}, CUDA available: {torch.cuda.is_available()}, device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
FIGDIR = Path("figures")
FIGDIR.mkdir(exist_ok=True)

torch 2.11.0+cu128, CUDA available: True, device: cuda
GPU: NVIDIA GeForce RTX 4080 Laptop GPU


## 2. Task definition and model choice


**Task:**
- **Goal:** train a small generative model that produces new images.
- **Data:** the Fashion-MNIST dataset (Xiao et al., 2017).
- **Method:** the model learns the dataset distribution, then generates new images from random noise.
- **Dataset section:** the dataset facts and the loading policy are in the dataset section below, starting next.


**Model:**
- **Type:** a conditional DCGAN.
- **Backbone:** Radford et al. (2016).
- **Conditioning:** the class label, in the way of Mirza and Osindero (2014).
- **Training improvement:** label smoothing, with real labels at `0.9` and fake labels at `0.1`.
- **Variant chosen:** the two-sided form that Salimans et al. (2016) recommend.
- **Effect:** it keeps the discriminator from becoming overconfident.


**Why the conditional variant:**
- **Classes:** Fashion-MNIST has `10` recognizable product categories, from T-shirt/top to Ankle boot.
- **Experiment:** conditioning the generator on the class label turns "generate a shirt" into a controlled experiment.
- **Control:** one fixed noise vector holds the other inputs constant.
- **Reading:** the label is then the only variable per column, so the per-class grids show what the condition controls and where generation fails.
- **Alternative:** an unconditional GAN or a VAE would make this evaluation weaker, a point the summary returns to.


**Evaluation:**
- **Qualitative:** the qualitative evaluation later in this notebook reads the outputs as images. The items are the per-class grids, the real-vs-generated pairs, and the failure strip.
- **Quantitative:** the quantitative evaluation later in this notebook measures a per-class agreement score, a feature-space Frechet distance, and an intra-class diversity measure.
- **Reader:** every measurement passes through one separately trained evaluation classifier, independent of the generator in architecture, parameters, and optimization, and trained on the same real data (the practice of Odena et al., 2017).
- **Feature space:** the distance and the diversity readings use the classifier's `128`-dimensional penultimate features (Heusel et al., 2017).
- **Protocol:** a generation-protocol experiment measures why generated images run in eval mode.
- **Caveat:** the measurements share one feature space, and therefore its biases. Each metric's justification is stated where it is introduced.
